# Hotel Hybrid Test (Free APIs Only)

This notebook keeps the same usability and return format, but uses only free APIs:
1. Nominatim (OpenStreetMap) for destination geocoding.
2. Overpass API for nearby lodging discovery (hotel/hostel/guest_house).
3. Smart price estimation + synthetic fallback when needed.

No paid API keys required.


In [2]:
import math
import asyncio
from datetime import date, timedelta
import httpx

NOMINATIM_URL = 'https://nominatim.openstreetmap.org/search'
OVERPASS_URL = 'https://overpass-api.de/api/interpreter'
USER_AGENT = 'PlanMyTrip-HotelTest/1.0 (educational test notebook)'

print('Using free APIs: Nominatim + Overpass')


Using free APIs: Nominatim + Overpass


In [3]:
def _safe_int(v, default=0):
    try:
        return int(float(v))
    except Exception:
        return default

def _safe_float(v, default=0.0):
    try:
        return float(v)
    except Exception:
        return default

def _normalize_name(s):
    return ''.join(ch for ch in (s or '').lower() if ch.isalnum() or ch.isspace()).strip()

def _estimate_price_per_night(stars, lodging_type, budget_per_night):
    # Heuristic pricing in INR from free metadata (stars/type)
    base = budget_per_night if budget_per_night > 0 else 2500
    stars = _safe_int(stars, 0)
    lt = (lodging_type or '').lower()
    type_mult = 1.0
    if 'hostel' in lt:
        type_mult = 0.65
    elif 'guest_house' in lt or 'guest house' in lt:
        type_mult = 0.8
    elif 'hotel' in lt:
        type_mult = 1.1

    star_mult_map = {0: 0.9, 1: 0.8, 2: 0.9, 3: 1.0, 4: 1.35, 5: 1.8}
    star_mult = star_mult_map.get(stars, 1.0)
    est = int(max(800, base * type_mult * star_mult))
    return max(800, min(est, 30000))

def _synthetic_hotels(destination, budget, days):
    d = max(1, days)
    return [
        {'name': f'Budget Stay - {destination}', 'type': 'Budget Hotel', 'price_per_night': max(800, int(budget / d * 0.15)), 'rating': 3.8, 'source': 'synthetic_fallback'},
        {'name': f'Comfort Inn - {destination}', 'type': '3-Star Hotel', 'price_per_night': max(2000, int(budget / d * 0.30)), 'rating': 4.2, 'source': 'synthetic_fallback'},
        {'name': f'Premium Resort - {destination}', 'type': '4-Star Hotel', 'price_per_night': max(4000, int(budget / d * 0.50)), 'rating': 4.6, 'source': 'synthetic_fallback'},
        {'name': f'Heritage Homestay - {destination}', 'type': 'Homestay', 'price_per_night': max(1200, int(budget / d * 0.20)), 'rating': 4.4, 'source': 'synthetic_fallback'},
    ]

def _score_hotels(hotels, budget_per_night):
    ranked = []
    for h in hotels:
        price = _safe_float(h.get('price_per_night'), 0)
        rating = _safe_float(h.get('rating'), 3.8)
        reviews = _safe_float(h.get('user_ratings_total'), 0)
        fit = 1.0 if budget_per_night <= 0 or price <= 0 else max(0.0, 1.0 - min(abs(price - budget_per_night) / budget_per_night, 1.0))
        rating_score = max(0.0, min(rating / 5.0, 1.0))
        review_score = min(1.0, math.log(reviews + 1) / math.log(1001)) if reviews > 0 else 0.2
        free_source_bonus = 0.15 if str(h.get('source', '')).startswith('osm') else 0.0
        score = fit * 0.45 + rating_score * 0.35 + review_score * 0.10 + free_source_bonus
        if budget_per_night > 0 and price > budget_per_night * 1.5:
            score -= 0.2
        ranked.append({**h, 'recommendation_score': round(max(0.0, score), 3)})
    ranked.sort(key=lambda x: x['recommendation_score'], reverse=True)
    for i, h in enumerate(ranked):
        h['is_recommended'] = (i == 0)
    return ranked


In [4]:
async def fetch_destination_coordinates(destination):
    params = {
        'q': destination,
        'format': 'json',
        'limit': 1,
    }
    headers = {'User-Agent': USER_AGENT}
    async with httpx.AsyncClient(timeout=20) as client:
        r = await client.get(NOMINATIM_URL, params=params, headers=headers)
        data = r.json() if r.content else []
    if not data:
        return None
    return {'lat': float(data[0]['lat']), 'lng': float(data[0]['lon'])}

def _hotel_type_from_tags(tags):
    tourism = (tags.get('tourism') or '').lower()
    if tourism == 'hostel':
        return 'Hostel'
    if tourism in ('guest_house', 'guest house'):
        return 'Guest House'
    if tourism == 'hotel':
        return 'Hotel'
    return 'Lodging'

async def fetch_google_lodging(destination, api_key=None):
    # Kept same function name to preserve notebook usability, now uses free OSM stack.
    coords = await fetch_destination_coordinates(destination)
    if not coords:
        return []

    query = f'''
[out:json][timeout:25];
(
  node["tourism"~"hotel|hostel|guest_house"](around:12000,{coords["lat"]},{coords["lng"]});
  way["tourism"~"hotel|hostel|guest_house"](around:12000,{coords["lat"]},{coords["lng"]});
  relation["tourism"~"hotel|hostel|guest_house"](around:12000,{coords["lat"]},{coords["lng"]});
);
out center 80;
'''

    headers = {'User-Agent': USER_AGENT}
    async with httpx.AsyncClient(timeout=30) as client:
        r = await client.post(OVERPASS_URL, data={"data": query}, headers=headers)
        data = r.json() if r.content else {}

    elements = data.get('elements', []) if isinstance(data, dict) else []
    out = []
    for el in elements:
        tags = el.get('tags', {})
        name = tags.get('name')
        if not name:
            continue
        lat = el.get('lat') if el.get('lat') is not None else el.get('center', {}).get('lat')
        lng = el.get('lon') if el.get('lon') is not None else el.get('center', {}).get('lon')
        if lat is None or lng is None:
            continue
        stars = tags.get('stars') or tags.get('hotel:stars') or 0
        rating = min(4.8, 3.4 + (_safe_int(stars, 0) * 0.28)) if _safe_int(stars, 0) > 0 else 3.9
        out.append({
            'name': name,
            'rating': round(rating, 1),
            'user_ratings_total': 0,
            'price_level': _safe_int(stars, 2),
            'location': tags.get('addr:full') or tags.get('addr:street') or destination,
            'lat': float(lat),
            'lng': float(lng),
            'lodging_type': _hotel_type_from_tags(tags),
            'stars': _safe_int(stars, 0),
            'source': 'osm_overpass',
        })

    # Deduplicate by normalized name
    dedup = {}
    for h in out:
        dedup[_normalize_name(h['name'])] = h
    return list(dedup.values())[:12]

async def fetch_live_price_map(destination, checkin, checkout, url=None, rapidapi_key="", rapidapi_host=""):
    # Free-mode notebook: keep this hook for compatibility, return empty map.
    return {}


In [5]:
async def get_hotels_hybrid(destination, budget, days):
    # Same interface/output shape as before
    days_safe = max(1, int(days))
    budget_per_night = float(budget) / days_safe if budget else 0
    checkin = (date.today() + timedelta(days=14)).isoformat()
    checkout = (date.today() + timedelta(days=14 + days_safe)).isoformat()

    try:
        free_hotels = await fetch_google_lodging(destination, api_key=None)
        live_price_map = await fetch_live_price_map(destination, checkin, checkout, url=None)

        if free_hotels:
            enriched = []
            for h in free_hotels:
                n = _normalize_name(h.get('name', ''))
                live_price = live_price_map.get(n)
                price = live_price or _estimate_price_per_night(h.get('stars', h.get('price_level', 2)), h.get('lodging_type', ''), budget_per_night or 2500)
                h_type = h.get('lodging_type') or ('Hotel' if h.get('stars', 0) >= 3 else 'Budget Hotel')
                enriched.append({
                    **h,
                    'type': h_type,
                    'price_per_night': int(price),
                    'source': 'live_pricing' if live_price else h.get('source', 'osm_overpass'),
                })

            ranked = _score_hotels(enriched, budget_per_night)
            if len(ranked) < 4:
                fallback = _score_hotels(_synthetic_hotels(destination, budget, days_safe), budget_per_night)
                seen = {_normalize_name(x['name']) for x in ranked}
                for f in fallback:
                    if _normalize_name(f['name']) not in seen:
                        ranked.append(f)
                    if len(ranked) >= 4:
                        break
                ranked = _score_hotels(ranked, budget_per_night)
            return ranked
    except Exception as e:
        print('Free hybrid fetch failed, using fallback:', e)

    return _score_hotels(_synthetic_hotels(destination, budget, days_safe), budget_per_night)

# Example usage (same as before)
results = await get_hotels_hybrid(destination='Goa', budget=30000, days=3)
results[:5]


[{'name': 'Hotel Yuvraj',
  'rating': 3.9,
  'user_ratings_total': 0,
  'price_level': 0,
  'location': 'Goa',
  'lat': 15.22066,
  'lng': 74.08627,
  'lodging_type': 'Hotel',
  'stars': 0,
  'source': 'osm_overpass',
  'type': 'Hotel',
  'price_per_night': 9900,
  'recommendation_score': 0.888,
  'is_recommended': True},
 {'name': 'Sai Raj Hotel',
  'rating': 3.9,
  'user_ratings_total': 0,
  'price_level': 0,
  'location': 'Goa',
  'lat': 15.21707,
  'lng': 74.08652,
  'lodging_type': 'Hotel',
  'stars': 0,
  'source': 'osm_overpass',
  'type': 'Hotel',
  'price_per_night': 9900,
  'recommendation_score': 0.888,
  'is_recommended': False},
 {'name': 'Chatrapati Hotel',
  'rating': 3.9,
  'user_ratings_total': 0,
  'price_level': 0,
  'location': 'Goa',
  'lat': 15.2575414,
  'lng': 74.1084987,
  'lodging_type': 'Hotel',
  'stars': 0,
  'source': 'osm_overpass',
  'type': 'Hotel',
  'price_per_night': 9900,
  'recommendation_score': 0.888,
  'is_recommended': False},
 {'name': 'Delfin